# 06c · `aede` en grilla de anchos, pesos y maxcl (iteración 3b, tareas D4 y D5)

**Objetivo.** Correr la EPCA del asesor (`aede`, port de Python `src/camiz_port.py`, D6) sobre la
grilla declarada de antemano en D10 (`decisiones.qmd`) y en el plan de la iteración 3b
(`asesoria/2026-10-21_plan-iteracion-3b.md`, tareas D4 y D5), en las dos vías (D1 y D2):

- anchos $n_w$ ∈ {24, 36, 48, 60} meses, paso de un mes, tres dimensiones;
- pesos uniformes y gaussianos (`nsd = 3`);
- Fisher sobre λ1 con maxcl ∈ {10, 15, 20};
- criterios: codo por caída máxima y Calinski-Harabász con $k$ y con $k-1$ grados de libertad
  (D11: el CH se informa y no selecciona).

La grilla se lee como **multiescala**, no como búsqueda de la ventana «correcta» (D10), y cada
$k^*$ se leerá contra el nulo de estructura constante (notebook 07b), no contra los episodios.

**Paridad (D6).** La referencia es la versión en R (`R/camiz_port.R`, Rmd, bloque
`b3b-aede-grilla`), que escribe `R_aede_cortes.csv` en formato largo. Si existe, este notebook
lo compara con el suyo ($k$ y fechas de corte exactos) y falla si difieren; la tabla de paridad
del Rmd hace la comparación inversa. Las filas con `nw = 60` y `maxcl = 10` son las de la
iteración 3a.

**Convención de fechado.** La fecha de cada ventana es la de su fila central, $i + n_w/2$
(1-based), como `rownames(serie)[(1 + nw/2):(wn + nw/2)]` en `aede_r`.

**Semillas.** `aede` y Fisher son determinísticos. El único sorteo es el del umbral de Horn de
la sección 3 (D5): `SEMILLA = 2026`, con el generador PCG64 de numpy (distinto del de R; ver
la excepción de generadores de la iteración 3a).

**Salidas** (niveles en `data/results/niveles/`, transformadas en `data/results/`):
`06c_aede_cortes.csv` y, en la sección 3, `06c_aede_valores_propios.csv`,
`06c_aede_valores_propios_resumen.csv` y `06c_aede_cruces.csv`.

In [1]:
# ---- Configuración --------------------------------------------------------------
import sys
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / '_quarto.yml').exists():
    RAIZ = RAIZ.parent
assert (RAIZ / '_quarto.yml').exists(), 'No se encontró la raíz del repositorio'
sys.path.insert(0, str(RAIZ))
from src import camiz_port as cp
from src.datos import cargar_vias

SEMILLA = 2026
NW_GRILLA = (24, 36, 48, 60)
PESOS = {'uniformes': 0, 'gaussianos_nsd3': 3}
MAXCL_GRILLA = (10, 15, 20)
NDIM = 3
CRITERIOS = [('codo (caída máxima)', 'k_codo', 'cortes_codo'),
             ('Calinski-Harabász k gl (aede)', 'k_ch', 'cortes_ch'),
             ('Calinski-Harabász k-1 gl (habitual)', 'k_ch1', 'cortes_ch1')]
DIR_T = RAIZ / 'data' / 'results'
DIR_N = DIR_T / 'niveles'

X, Xt = cargar_vias(RAIZ)
VIAS = {'niveles': (X, DIR_N), 'transformadas': (Xt, DIR_T)}
for via, (M, _) in VIAS.items():
    print(via, M.shape, M.index[0], '→', M.index[-1])

niveles (228, 17) 2007-01 → 2025-12
transformadas (227, 17) 2007-02 → 2025-12


## 1. Grilla de `aede` (D4)

In [2]:
corridas = {}      # (vía, nw, pesos) -> salida de aede con maxcl = 10 (para la sección 3)
cortes = {}
for via, (M, d) in VIAS.items():
    filas = []
    for nw in NW_GRILLA:
        for pesos, nsd in PESOS.items():
            for maxcl in MAXCL_GRILLA:
                a = cp.aede(M, nw=nw, ndim=NDIM, nsd=nsd, maxcl=maxcl)
                r = cp.resumen_aede(a)
                if maxcl == MAXCL_GRILLA[0]:
                    corridas[(via, nw, pesos)] = a
                for crit, kk, cc in CRITERIOS:
                    filas.append({'nw': nw, 'maxcl': maxcl, 'pesos': pesos, 'criterio': crit, 'k': r[kk],
                                  'k_en_tope': r[kk] == maxcl, 'cortes': ' '.join(r[cc])})
    tab = pd.DataFrame(filas)
    tab.to_csv(d / '06c_aede_cortes.csv', index=False)
    cortes[via] = tab
    print(via, len(tab), 'filas')

niveles 72 filas


transformadas 72 filas


### 1.1 Comparación con R y no regresión

Con `nw = 60` y `maxcl = 10`, las filas deben ser las del `999_port_aede_cortes.csv` de la
iteración 3a (que reproducía el `R_aede_cortes.csv` de entonces).

In [3]:
for via, (M, d) in VIAS.items():
    py = pd.read_csv(d / '06c_aede_cortes.csv', keep_default_na=False)
    viejo = pd.read_csv(d / '999_port_aede_cortes.csv', keep_default_na=False)
    s60 = py[(py.nw == 60) & (py.maxcl == 10)].reset_index(drop=True)
    assert (s60[['pesos', 'criterio', 'k', 'cortes']].astype(str).to_numpy()
            == viejo[['pesos', 'criterio', 'k', 'cortes']].astype(str).to_numpy()).all(), via
    f_r = d / 'R_aede_cortes.csv'
    ref = pd.read_csv(f_r, keep_default_na=False)
    if 'nw' in ref.columns:
        clave = lambda t: [f'{a} {b} {c} {e} {k} {str(t_).upper()} {ct}' for a, b, c, e, k, t_, ct in
                           zip(t.nw, t.maxcl, t.pesos, t.criterio, t.k, t.k_en_tope, t.cortes)]
        n_dif = sum(a != b for a, b in zip(clave(ref), clave(py)))
        assert len(ref) == len(py) and n_dif == 0, (via, n_dif)
        print(f'{via}: coincide con {f_r.name} ({len(py)} filas) y reproduce las filas de 3a')
    else:
        print(f'{via}: {f_r.name} aún en el formato de 3a (ejecutar el Rmd); reproduce las filas de 3a')

niveles: coincide con R_aede_cortes.csv (72 filas) y reproduce las filas de 3a
transformadas: coincide con R_aede_cortes.csv (72 filas) y reproduce las filas de 3a


In [4]:
# Vista compacta: k por criterio y maxcl (* = en el tope)
for via in VIAS:
    t = cortes[via].copy()
    t['celda'] = t.k.astype(str) + np.where(t.k_en_tope, '*', '')
    print(via)
    display(t.pivot_table(index=['nw', 'pesos', 'criterio'], columns='maxcl', values='celda', aggfunc='first'))

niveles


maxcl                                                    10   15   20
nw pesos           criterio                                          
24 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    6    6    6
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    6  15*  20*
                   codo (caída máxima)                    6    6    6
36 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2  15*  20*
                   codo (caída máxima)                    2    2    2
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2    2    2
                   codo (caída máxima)                    2    2    2
48 gaussianos_nsd3 Calinski-Harabász k gl (aede)          3  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2    2    2
                   codo (caída máxima)                    2    2    2
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2    2    2
                   codo (caída máxima)                    2    2    2
60 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*   14  20*
                   codo (caída máxima)                    2    2    2
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    2    2    2
                   codo (caída máxima)                    2    2    2

transformadas


maxcl                                                    10   15   20
nw pesos           criterio                                          
24 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*   14  20*
                   Calinski-Harabász k-1 gl (habitual)  10*   14  20*
                   codo (caída máxima)                    4   10   10
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    4    4    4
36 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    3    3    3
   uniformes       Calinski-Harabász k gl (aede)          9  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    9    9  20*
                   codo (caída máxima)                    3    3    3
48 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    3    3    3
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)    7  15*  20*
                   codo (caída máxima)                    4    4    4
60 gaussianos_nsd3 Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    4    4    4
   uniformes       Calinski-Harabász k gl (aede)        10*  15*  20*
                   Calinski-Harabász k-1 gl (habitual)  10*  15*  20*
                   codo (caída máxima)                    4    4    4

## 2. Valores propios por ventana frente a Kaiser y Horn (D5; apunte 7 en su versión útil, D14)

Para cada ancho y esquema de pesos: λ1–λ3 de cada ventana; el $n$ efectivo de Kish de los pesos,
$n_\omega = 1/\sum_i \omega_i^2$ (pesos normalizados a suma 1), que depende solo de los pesos;
el umbral de Kaiser (1) y el p95 de Horn para λ1–λ3 con $n = \lceil n_\omega \rceil$ (1 000
simulaciones gaussianas de 17 series, `SEMILLA`). El techo se toma con una holgura de $10^{-9}$
para que el redondeo de $1/\sum \omega_i^2$ no convierta $n_\omega = n_w$ en $n_w + 1$.

**Cruce λ2 ↔ λ3:** la ventana $i$ registra un cruce si su eje 2 es más congruente (|Tucker|)
con el eje 3 de la ventana $i-1$ que con el eje 2 (cargas alineadas de `aede`,
`correlationlsq`).

**Umbral de Horn compartido.** R y Python sortean con generadores distintos (excepción de
generadores de la iteración 3a). Con 1 000 simulaciones los dos umbrales difieren hasta en unas
centésimas, y como muchas ventanas quedan cerca del umbral, los conteos también difieren. Por eso
los conteos oficiales usan el umbral que escribe el Rmd (`R_aede_horn_umbrales.csv`), como los
índices bootstrap compartidos de 3a; el umbral propio de numpy y su conteo se informan aparte
(`horn_p95_numpy`, `bajo_horn_numpy`). Si el archivo de R no existe, se usa el de numpy y se
marca en `fuente_umbral_horn`.

**Empates con Kaiser.** Con $n_w = 24$ la tasa de referencia es constante en algunas ventanas
(2011–2013): `aede` la deja aislada y un valor propio vale 1 salvo redondeo (en R y en Python el
último bit difiere). Se cuenta «bajo Kaiser» si $\lambda < 1 - 10^{-9}$ y aparte las ventanas a
menos de $10^{-9}$ de 1 (`en_umbral_kaiser`); `serie_constante` cuenta las ventanas con alguna
serie constante.

In [5]:
from src.multivariado import umbral_horn, n_kish, tucker

P = X.shape[1]
TOL_KAISER = 1e-9
cache_horn = {}
def horn_numpy(n_h):
    if n_h not in cache_horn:
        cache_horn[n_h] = umbral_horn(n_h, P, n_sim=1000, percentil=0.95, semilla=SEMILLA)
    return cache_horn[n_h]

f_horn = DIR_T / 'R_aede_horn_umbrales.csv'
HORN_R = pd.read_csv(f_horn) if f_horn.exists() else None
def horn_compartido(n_h):
    # Umbral p95 de R para λ1..λ3 con n_h observaciones; si no existe, el de numpy
    if HORN_R is not None and (HORN_R.n_horn == n_h).any():
        return HORN_R[HORN_R.n_horn == n_h].sort_values('eje').horn_p95.to_numpy()[:3], 'R'
    return horn_numpy(n_h)[:3], 'numpy'

def constante_ventana(M, nw):
    A = M.to_numpy()
    return np.array([(A[i:i + nw] == A[i]).all(axis=0).any() for i in range(len(A) - nw + 1)])

vpv = {}
for via, (M, d) in VIAS.items():
    vp_l, res_l, cr_l = [], [], []
    for nw in NW_GRILLA:
        for pesos in PESOS:
            a = corridas[(via, nw, pesos)]
            ev = a['eigenvalues'].to_numpy()
            n_om = n_kish(a['weights'])
            n_h = int(np.ceil(n_om - 1e-9))
            hu, fuente = horn_compartido(n_h)
            hn = horn_numpy(n_h)[:3]
            # Criterio 5 del plan: λ1 + λ2 + λ3 <= 17 y n_omega = nw con pesos uniformes
            assert (ev.sum(axis=1) <= P + 1e-9).all()
            assert pesos != 'uniformes' or abs(n_om - nw) < 1e-9
            fv = list(a['eigenvalues'].index)
            vp_l.append(pd.DataFrame({'fecha': fv, 'nw': nw, 'pesos': pesos, 'lambda1': ev[:, 0], 'lambda2': ev[:, 1],
                                      'lambda3': ev[:, 2], 'n_omega': n_om, 'n_horn': n_h, 'umbral_kaiser': 1.0,
                                      'horn_p95_1': hu[0], 'horn_p95_2': hu[1], 'horn_p95_3': hu[2],
                                      'horn_p95_numpy_1': hn[0], 'horn_p95_numpy_2': hn[1], 'horn_p95_numpy_3': hn[2]}))
            C = a['correlationlsq']
            t22 = np.array([abs(tucker(C[i, :, 1], C[i - 1, :, 1])) for i in range(1, len(ev))])
            t23 = np.array([abs(tucker(C[i, :, 1], C[i - 1, :, 2])) for i in range(1, len(ev))])
            cruce = np.nonzero(t23 > t22)[0] + 1
            cr_l.append(pd.DataFrame({'fecha': [fv[i] for i in cruce], 'nw': nw, 'pesos': pesos,
                                      'tucker_22': t22[cruce - 1], 'tucker_23': t23[cruce - 1],
                                      'brecha_23': ev[cruce, 1] - ev[cruce, 2]}))
            res_l.append(pd.DataFrame({'nw': nw, 'pesos': pesos, 'eje': [1, 2, 3], 'ventanas': len(ev), 'n_omega': n_om,
                                       'n_horn': n_h, 'bajo_kaiser': (ev < 1 - TOL_KAISER).sum(axis=0),
                                       'en_umbral_kaiser': (np.abs(ev - 1) <= TOL_KAISER).sum(axis=0),
                                       'serie_constante': int(constante_ventana(M, nw).sum()),
                                       'bajo_horn': (ev < hu).sum(axis=0), 'horn_p95': hu, 'fuente_umbral_horn': fuente,
                                       'bajo_horn_numpy': (ev < hn).sum(axis=0), 'horn_p95_numpy': hn,
                                       'cruces_2_3': len(cruce)}))
    v = {'vp': pd.concat(vp_l, ignore_index=True), 'resumen': pd.concat(res_l, ignore_index=True),
         'cruces': pd.concat(cr_l, ignore_index=True)}
    v['vp'].to_csv(d / '06c_aede_valores_propios.csv', index=False)
    v['resumen'].to_csv(d / '06c_aede_valores_propios_resumen.csv', index=False)
    v['cruces'].to_csv(d / '06c_aede_cruces.csv', index=False)
    vpv[via] = v
    print(via, len(v['vp']), 'ventanas ·', len(v['cruces']), 'cruces λ2 ↔ λ3')
    display(v['resumen'])

niveles 1496 ventanas · 16 cruces λ2 ↔ λ3


,nw,pesos,eje,ventanas,n_omega,n_horn,bajo_kaiser,en_umbral_kaiser,serie_constante,bajo_horn,horn_p95,fuente_umbral_horn,bajo_horn_numpy,horn_p95_numpy,cruces_2_3
0,24,uniformes,1,205,24.000000,24,0,0,7,0,3.290383,R,0,3.276902,3
1,24,uniformes,2,205,24.000000,24,0,0,7,117,2.671620,R,117,2.672336,3
2,24,uniformes,3,205,24.000000,24,71,7,7,198,2.265949,R,198,2.251164,3
3,24,gaussianos_nsd3,1,205,18.237952,19,0,0,7,0,3.677434,R,0,3.677137,4
4,24,gaussianos_nsd3,2,205,18.237952,19,0,0,7,123,2.906275,R,123,2.889175,4
5,24,gaussianos_nsd3,3,205,18.237952,19,48,5,7,199,2.453914,R,199,2.433131,4
6,36,uniformes,1,193,36.000000,36,0,0,0,0,2.767496,R,0,2.774529,2
7,36,uniformes,2,193,36.000000,36,1,0,0,53,2.299191,R,53,2.314762,2
8,36,uniformes,3,193,36.000000,36,51,0,0,180,2.007699,R,180,2.003775,2
9,36,gaussianos_nsd3,1,193,27.626910,28,0,0,0,0,3.062797,R,0,3.075798,1


transformadas 1488 ventanas · 26 cruces λ2 ↔ λ3


,nw,pesos,eje,ventanas,n_omega,n_horn,bajo_kaiser,en_umbral_kaiser,serie_constante,bajo_horn,horn_p95,fuente_umbral_horn,bajo_horn_numpy,horn_p95_numpy,cruces_2_3
0,24,uniformes,1,204,24.000000,24,0,0,6,7,3.290383,R,7,3.276902,9
1,24,uniformes,2,204,24.000000,24,0,0,6,51,2.671620,R,51,2.672336,9
2,24,uniformes,3,204,24.000000,24,0,0,6,125,2.265949,R,121,2.251164,9
3,24,gaussianos_nsd3,1,204,18.237952,19,0,0,6,33,3.677434,R,33,3.677137,2
4,24,gaussianos_nsd3,2,204,18.237952,19,0,0,6,56,2.906275,R,52,2.889175,2
5,24,gaussianos_nsd3,3,204,18.237952,19,0,0,6,159,2.453914,R,156,2.433131,2
6,36,uniformes,1,192,36.000000,36,0,0,0,0,2.767496,R,0,2.774529,6
7,36,uniformes,2,192,36.000000,36,0,0,0,22,2.299191,R,23,2.314762,6
8,36,uniformes,3,192,36.000000,36,0,0,0,67,2.007699,R,67,2.003775,6
9,36,gaussianos_nsd3,1,192,27.626910,28,0,0,0,0,3.062797,R,0,3.075798,2


### 2.1 Comparación con R

λ1–λ3 y $n_\omega$ con tolerancia $10^{-10}$; conteos bajo Kaiser, cruces y conteos bajo Horn
(con el umbral compartido) exactos.

In [6]:
for via, (M, d) in VIAS.items():
    f_r = d / 'R_aede_valores_propios.csv'
    if not f_r.exists():
        print(f'{via}: aún no existe {f_r.name} (ejecutar el Rmd)')
        continue
    v = vpv[via]
    ref = pd.read_csv(f_r)
    m = v['vp'].merge(ref, on=['nw', 'pesos', 'fecha'], suffixes=('', '_r'), validate='one_to_one')
    assert len(m) == len(ref) == len(v['vp'])
    dif = max(float(np.max(np.abs(m[c] - m[c + '_r']))) for c in ['lambda1', 'lambda2', 'lambda3', 'n_omega'])
    assert dif < 1e-10, (via, dif)
    rr = pd.read_csv(d / 'R_aede_valores_propios_resumen.csv')
    mr = v['resumen'].merge(rr, on=['nw', 'pesos', 'eje'], suffixes=('', '_r'))
    assert (v['resumen'].fuente_umbral_horn == 'R').all(), 'umbral de Horn sin compartir: ejecutar primero el Rmd'
    for c in ['ventanas', 'n_horn', 'bajo_kaiser', 'en_umbral_kaiser', 'serie_constante', 'cruces_2_3', 'bajo_horn']:
        assert (mr[c] == mr[c + '_r']).all(), (via, c, mr.loc[mr[c] != mr[c + '_r'], ['nw', 'pesos', 'eje', c, c + '_r']])
    dif_h = float(np.max(np.abs(mr.horn_p95_numpy - mr.horn_p95_r)))
    n_dif_numpy = int((mr.bajo_horn_numpy != mr.bajo_horn).sum())
    cr = pd.read_csv(d / 'R_aede_cruces.csv')
    assert list(zip(cr.nw, cr.pesos, cr.fecha)) == list(zip(v['cruces'].nw, v['cruces'].pesos, v['cruces'].fecha)), via
    print(f'{via}: coincide con R (dif. máx. λ y n_omega {dif:.1e}; conteos y cruces exactos; '
          f'umbral de Horn compartido; el de numpy difiere hasta {dif_h:.3f} y cambia el conteo en {n_dif_numpy} de {len(mr)} filas)')

niveles: coincide con R (dif. máx. λ y n_omega 7.5e-14; conteos y cruces exactos; umbral de Horn compartido; el de numpy difiere hasta 0.035 y cambia el conteo en 3 de 24 filas)
transformadas: coincide con R (dif. máx. λ y n_omega 6.4e-14; conteos y cruces exactos; umbral de Horn compartido; el de numpy difiere hasta 0.035 y cambia el conteo en 10 de 24 filas)


**Lectura.** Este notebook no interpreta: deja la grilla de cortes y los valores propios por
ventana como insumo de la fase 1 (D10, D11 y D13 de `decisiones.qmd`), donde se leen contra el
nulo de estructura constante del notebook 07b.